# GraMM-RAG — MMLongBench-Doc End-to-End Implementation

**Full end-to-end notebook** for the dissertation.  
Dataset: `yubo2333/MMLongBench-Doc` — 1,091 QA records, 135 PDFs, avg 47.5 pages.  
Generation model: **Qwen2.5-VL-72B-Instruct** (Together.ai).  
Metric: **F1** (token overlap with gold answer) + ANLS + Accuracy.

| Phase | Description |
|---|---|
| 0 | Setup & configuration |
| 1 | Load all 1,091 QA records + EDA |
| 1.5 | Exploratory data analysis — corpus stats, charts, tables |
| 2 | Verify PDF availability from local folder |
| 3 | Parse PDFs with MinerU + temporal annotation |
| 4 | Compute embeddings (E5-Mistral-7B + SigLIP) + build PyG graphs |
| 5 | Train HGT with evidence-guided InfoNCE loss (50 epochs) |
| 6 | Train query router (DeBERTa-v3-base, 3 epochs) |
| 7 | Tune reward function (α, β, λ, τ grid search) |
| 8 | Flat-vector RAG baseline (FAISS + Qwen2.5-VL-72B) |
| 9 | GraMM-RAG full evaluation (3 seeds × 1,091 questions) |
| 10 | Results comparison table |

**Prerequisites:** GPU with ≥16 GB VRAM, `TOGETHER_API_KEY`, `OPENAI_API_KEY`, PDFs in `data/mmlongbench/pdfs/`.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# PHASE 0 — Setup & Configuration
# ═══════════════════════════════════════════════════════════════════════════

N_PILOT = None     # set an int (e.g. 100) for a quick pilot run

SEED          = 42
HF_REPO       = 'yubo2333/MMLongBench-Doc'
HGT_EPOCHS    = 5  if N_PILOT else 50
ROUTER_EPOCHS = 1  if N_PILOT else 3

# ─── Retrieval / generator config (ported from mpdocvqa locked-best) ───────
GEN_TOP_N          = 8           # cap for node-mode (PAGE_CONTEXT=False)
PAGE_CONTEXT       = True        # feed top-N pages full text in reading order
PAGE_CONTEXT_N     = 2           # top-1 + top-2 pages
PAGE_CONTEXT_CHARS = 8000        # build_prompt truncation in page mode
PAGE_SCORE_MODE    = 'first_seen'  # 'first_seen' | 'mean_topk' | 'sum'
PAGE_SCORE_TOPK    = 3
USE_CROSS_ENCODER   = False      # bi-encoder default (empirically best on text)
CROSS_ENCODER_MODEL = 'cross-encoder/ms-marco-MiniLM-L-6-v2'
RERANK_POOL         = 20         # vector-path candidates before reranking

import sys, pathlib
ROOT = pathlib.Path('.').resolve()          # flat repo root (src/ alongside notebook)
sys.path.insert(0, str(ROOT))

DATA_DIR    = ROOT / 'data'       / 'mmlongbench'
PDF_DIR     = DATA_DIR / 'pdfs'
# Derived artefacts live at the repo root (flat, self-contained layout).
PARSED_DIR  = ROOT / 'parsed'
EMB_DIR     = ROOT / 'embeddings'
GRAPH_DIR   = ROOT / 'graphs'
MODEL_DIR   = ROOT / 'results' / 'models'
RESULTS_DIR = ROOT / 'results'

# ─── Fallback path variables (defined early so later phases never NameError)
hgt_save_path   = MODEL_DIR / 'hgt_mmlb'  / 'best_model.pt'
reward_save     = MODEL_DIR / 'reward_mmlb.json'
router_save_dir = MODEL_DIR / 'router_mmlb' / 'best_model'
vector_out_path = RESULTS_DIR / 'baseline_vector_mmlb.json'
faiss_indices     = {}       # populated in Phase 8
gramm_results     = {}       # populated in Phase 9
doc_id_to_pdf     = {}       # populated in Phase 2 cell 2
parseable_doc_ids = []       # populated in Phase 2 cell 2

print(f'ROOT:   {ROOT}')
print(f'DEVICE: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print(f'N_PILOT = {N_PILOT}  (full 1,091-question run)')

In [ ]:
import json, random, ast, logging, os
from pathlib import Path
from collections import Counter, defaultdict
import torch

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
logger = logging.getLogger('mmlb_e2e')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

# Load API keys from .env file (works for both Docker and bare Python)
try:
    from dotenv import load_dotenv
    _loaded = load_dotenv(dotenv_path=ROOT / '.env', override=False)
    if not _loaded:
        load_dotenv(dotenv_path=pathlib.Path('.env'), override=False)
    print('API keys loaded from .env')
except ImportError:
    print('python-dotenv not installed — reading keys from shell environment only')
    print('  Install with: pip install python-dotenv')

if not os.environ.get('TOGETHER_API_KEY'):
    print('⚠ TOGETHER_API_KEY not set — generation will use extractive fallback')
if not os.environ.get('OPENAI_API_KEY'):
    print('⚠ OPENAI_API_KEY not set — KG triplets will be empty (safe to proceed)')

In [ ]:
for d in [DATA_DIR, PDF_DIR, PARSED_DIR, EMB_DIR, GRAPH_DIR,
          MODEL_DIR / 'hgt_mmlb',
          MODEL_DIR / 'router_mmlb',
          RESULTS_DIR / 'figures']:
    d.mkdir(parents=True, exist_ok=True)
print('All directories ready.')

## Phase 1 — Load Data

Load all 1,091 MMLongBench-Doc QA records. Normalise string-encoded fields
(`evidence_pages`, `evidence_sources`) to Python lists for downstream use.
Set `N_PILOT` to an integer (e.g. 100) at the top of Phase 0 for a quick smoke-test.

In [ ]:
def parse_str_list(s):
    """Parse strings like '[5]' or '[19, 20]' to int/str lists."""
    if not s or s in ('[]', 'None', None):
        return []
    try:
        result = ast.literal_eval(s) if isinstance(s, str) else s
        return list(result) if isinstance(result, (list, tuple)) else []
    except Exception:
        return []

all_questions = json.loads((DATA_DIR / 'mmlongbench_doc.json').read_text(encoding='utf-8'))

for q in all_questions:
    q['_evidence_pages']   = [int(p) for p in parse_str_list(q.get('evidence_pages', '[]'))]
    q['_evidence_sources'] = parse_str_list(q.get('evidence_sources', '[]'))
    # NOTE: all 1,091 questions have non-null answers. answer_format='None' (22.4%)
    # means free-form format, NOT unanswerable.
    q['_unanswerable']     = q.get('answer') is None or str(q.get('answer', '')).strip() == ''
    q['_freeform_fmt']     = q.get('answer_format') == 'None'   # 22.4% free-form
    q['_n_ev_pages']       = len(q['_evidence_pages'])
    q['_q_words']          = len(q['question'].split())
    q['_a_words']          = len(str(q.get('answer') or '').split())

random.seed(SEED)
pilot_questions = random.sample(all_questions, N_PILOT) if N_PILOT else all_questions
pilot_doc_ids   = sorted({q['doc_id'] for q in pilot_questions})

print(f'Total QA records  : {len(all_questions):,}')
print(f'Pilot questions   : {len(pilot_questions):,}')
print(f'Unique pilot docs : {len(pilot_doc_ids)}')

In [ ]:
import pandas as pd

type_counts = Counter(q['doc_type'] for q in pilot_questions)
fmt_counts  = Counter(q.get('answer_format', 'Unknown') for q in pilot_questions)
n_unanswerable = sum(q['_unanswerable'] for q in pilot_questions)
n_crosspage    = sum(q['_n_ev_pages'] > 1 for q in pilot_questions)
n_visual = sum(
    any(s in ('Chart', 'Figure', 'Table') for s in q['_evidence_sources'])
    for q in pilot_questions)

print(f'=== Pilot Set Statistics (N={len(pilot_questions)}) ===')
print(f'  Unanswerable   : {n_unanswerable} ({100*n_unanswerable/len(pilot_questions):.1f}%)')
print(f'  Cross-page     : {n_crosspage} ({100*n_crosspage/len(pilot_questions):.1f}%)')
print(f'  Visual evidence: {n_visual} ({100*n_visual/len(pilot_questions):.1f}%)')
print()
print('Doc types (top 5):', dict(type_counts.most_common(5)))
print('Answer formats:', dict(fmt_counts))

## Phase 1.5 — Exploratory Data Analysis

Comprehensive descriptive statistics, charts, and tables for the full
MMLongBench-Doc corpus and the pilot subset.  
All figures are saved to `results/figures/`.

| Analysis | Description |
|---|---|
| Corpus overview | Key stats table (full vs pilot) |
| Doc type distribution | Bar chart — top document categories |
| Answer format distribution | Horizontal bar chart |
| Evidence pages distribution | Histogram — pages required per question |
| Questions per document | Histogram — workload distribution across PDFs |
| Evidence sources | Bar chart — Chart / Table / Figure / Text breakdown |
| Question / answer length | Histogram — word counts |
| Unanswerable by category | Stacked bar — answerable vs unanswerable by doc_type |
| Cross-tabulation | Heatmap — doc_type × answer_format |
| Sample QA pairs | Table — 5 example questions per answer format |

In [ ]:
import matplotlib
matplotlib.use('Agg')   # non-interactive backend for nbconvert
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import numpy as np

sns.set_theme(style='whitegrid', palette='muted')
FIG_DIR = RESULTS_DIR / 'figures'
FIG_DIR.mkdir(exist_ok=True)

# ─── Full corpus derived stats ────────────────────────────────────────────
N_ALL   = len(all_questions)
N_DOCS  = len({q['doc_id'] for q in all_questions})
N_UNANS    = sum(q['_unanswerable']  for q in all_questions)
N_FREEFORM = sum(q['_freeform_fmt']  for q in all_questions)
N_CROSS    = sum(q['_n_ev_pages'] > 1 for q in all_questions)
N_VIS      = sum(
    any(s in ('Chart','Figure','Table') for s in q['_evidence_sources'])
    for q in all_questions)
MEAN_EV = np.mean([q['_n_ev_pages'] for q in all_questions])
MEAN_QW = np.mean([q['_q_words']    for q in all_questions])
MEAN_AW = np.mean([q['_a_words']    for q in all_questions])

p_freeform = sum(q['_freeform_fmt'] for q in pilot_questions)

overview = pd.DataFrame({
    'Metric': [
        'Total QA records', 'Unique PDFs',
        'Null/empty answer (truly unanswerable)',
        'Free-form format (answer_format=None)',
        'Cross-page (>1 evidence page)',
        'Visual evidence (Chart/Table/Figure)',
        'Mean evidence pages',
        'Mean question length (words)',
        'Mean answer length (words)',
    ],
    'Full corpus (1,091)': [
        f'{N_ALL:,}', f'{N_DOCS}',
        f'{N_UNANS} ({100*N_UNANS/N_ALL:.1f}%)',
        f'{N_FREEFORM} ({100*N_FREEFORM/N_ALL:.1f}%)',
        f'{N_CROSS} ({100*N_CROSS/N_ALL:.1f}%)',
        f'{N_VIS} ({100*N_VIS/N_ALL:.1f}%)',
        f'{MEAN_EV:.2f}',
        f'{MEAN_QW:.1f}',
        f'{MEAN_AW:.1f}',
    ],
    f'Pilot (N={len(pilot_questions)})': [
        f'{len(pilot_questions)}', f'{len(pilot_doc_ids)}',
        f'{n_unanswerable} ({100*n_unanswerable/len(pilot_questions):.1f}%)',
        f'{p_freeform} ({100*p_freeform/len(pilot_questions):.1f}%)',
        f'{n_crosspage} ({100*n_crosspage/len(pilot_questions):.1f}%)',
        f'{n_visual} ({100*n_visual/len(pilot_questions):.1f}%)',
        f"{np.mean([q['_n_ev_pages'] for q in pilot_questions]):.2f}",
        f"{np.mean([q['_q_words'] for q in pilot_questions]):.1f}",
        f"{np.mean([q['_a_words'] for q in pilot_questions]):.1f}",
    ],
})
print('=== Corpus Overview ===')
print(overview.to_string(index=False))

In [ ]:
# ─── Figure 1: Doc type and answer format distributions ──────────────────
all_type_counts = Counter(q['doc_type'] for q in all_questions)
all_fmt_counts  = Counter(q.get('answer_format','Unknown') for q in all_questions)

# Shorten long doc_type labels
def shorten(s, n=30):
    return s[:n] + '…' if len(s) > n else s

top_types = all_type_counts.most_common(10)
type_labels = [shorten(t, 28) for t, _ in top_types]
type_vals   = [c for _, c in top_types]

fmt_items   = sorted(all_fmt_counts.items(), key=lambda x: -x[1])
fmt_labels  = [f for f, _ in fmt_items]
fmt_vals    = [c for _, c in fmt_items]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('MMLongBench-Doc — Document & Answer Distributions (full corpus)', fontsize=13)

# Doc type bar chart
bars = axes[0].barh(type_labels[::-1], type_vals[::-1],
                    color=sns.color_palette('muted', len(type_vals)))
axes[0].set_xlabel('Number of questions')
axes[0].set_title('Top 10 Document Types')
for bar, val in zip(bars, type_vals[::-1]):
    axes[0].text(bar.get_width() + 2, bar.get_y() + bar.get_height()/2,
                 str(val), va='center', fontsize=8)

# Answer format horizontal bar
colors = sns.color_palette('Set2', len(fmt_labels))
axes[1].barh(fmt_labels[::-1], fmt_vals[::-1], color=colors)
axes[1].set_xlabel('Number of questions')
axes[1].set_title('Answer Format Distribution')
for i, (label, val) in enumerate(zip(fmt_labels[::-1], fmt_vals[::-1])):
    axes[1].text(val + 2, i, f'{val} ({100*val/N_ALL:.1f}%)', va='center', fontsize=9)

plt.tight_layout()
fig.savefig(str(FIG_DIR / 'fig1_type_format_dist.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {FIG_DIR}/fig1_type_format_dist.png')

In [ ]:
# ─── Figure 2: Evidence pages and questions-per-doc histograms ───────────
ev_page_counts = [q['_n_ev_pages'] for q in all_questions]
q_per_doc = Counter(q['doc_id'] for q in all_questions)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
fig.suptitle('MMLongBench-Doc — Evidence & Coverage Analysis', fontsize=13)

# Evidence pages histogram
max_ev = max(ev_page_counts) if ev_page_counts else 5
bins_ev = range(0, min(max_ev + 2, 15))
axes[0].hist(ev_page_counts, bins=bins_ev, color='steelblue',
             edgecolor='white', linewidth=0.8, align='left')
axes[0].set_xlabel('Number of evidence pages per question')
axes[0].set_ylabel('Number of questions')
axes[0].set_title('Evidence Pages Required per Question')
axes[0].set_xticks(range(0, min(max_ev + 1, 14)))
# Annotate proportions
for n_pgs in range(0, min(max_ev + 1, 5)):
    cnt = ev_page_counts.count(n_pgs)
    if cnt > 0:
        axes[0].annotate(f'{100*cnt/N_ALL:.1f}%',
                         xy=(n_pgs, cnt), ha='center', va='bottom', fontsize=8)

# Questions per document histogram
qs_per_doc_vals = list(q_per_doc.values())
axes[1].hist(qs_per_doc_vals, bins=20, color='coral', edgecolor='white', linewidth=0.8)
axes[1].set_xlabel('Questions per PDF document')
axes[1].set_ylabel('Number of documents')
axes[1].set_title(f'Questions per Document (N={N_DOCS} PDFs)')
axes[1].axvline(np.mean(qs_per_doc_vals), color='navy', linestyle='--',
                linewidth=1.5, label=f'Mean = {np.mean(qs_per_doc_vals):.1f}')
axes[1].axvline(np.median(qs_per_doc_vals), color='darkred', linestyle=':',
                linewidth=1.5, label=f'Median = {np.median(qs_per_doc_vals):.0f}')
axes[1].legend(fontsize=9)

plt.tight_layout()
fig.savefig(str(FIG_DIR / 'fig2_evidence_coverage.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {FIG_DIR}/fig2_evidence_coverage.png')
print(f'\nEvidence pages summary:')
print(f'  0 (unanswerable or no annotation): {ev_page_counts.count(0):,}')
print(f'  1 (single-page):                   {ev_page_counts.count(1):,}')
print(f'  2+ (cross-page):                   {sum(c >= 2 for c in ev_page_counts):,}')
print(f'  Max evidence pages:                {max(ev_page_counts)}')
print(f'  Mean questions/doc:                {np.mean(qs_per_doc_vals):.1f}')
print(f'  Max questions/doc:                 {max(qs_per_doc_vals)}')

In [ ]:
# ─── Figure 3: Evidence sources and question/answer length distributions ──
all_sources = [s for q in all_questions for s in q['_evidence_sources']]
src_counts  = Counter(all_sources)

q_words_all = [q['_q_words'] for q in all_questions]
a_words_all = [q['_a_words'] for q in all_questions if not q['_unanswerable']]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
fig.suptitle('MMLongBench-Doc — Evidence Sources & Text Length Analysis', fontsize=13)

# Evidence sources bar chart
src_labels = [k for k, _ in src_counts.most_common()]
src_vals   = [v for _, v in src_counts.most_common()]
axes[0].bar(src_labels, src_vals,
            color=sns.color_palette('Set1', len(src_labels)))
axes[0].set_xlabel('Evidence source type')
axes[0].set_ylabel('Count (QA × sources)')
axes[0].set_title('Evidence Source Types')
for i, (lbl, v) in enumerate(zip(src_labels, src_vals)):
    axes[0].text(i, v + 2, str(v), ha='center', fontsize=9)

# Question word count histogram
axes[1].hist(q_words_all, bins=30, color='mediumseagreen', edgecolor='white')
axes[1].set_xlabel('Question length (words)')
axes[1].set_ylabel('Number of questions')
axes[1].set_title(f'Question Length Distribution\n(mean={np.mean(q_words_all):.1f}, '
                  f'median={np.median(q_words_all):.0f})')
axes[1].axvline(np.mean(q_words_all), color='red', linestyle='--', linewidth=1.5)

# Answer word count histogram (answerable only)
axes[2].hist(a_words_all, bins=30, color='mediumpurple', edgecolor='white')
axes[2].set_xlabel('Answer length (words)')
axes[2].set_ylabel('Number of answers')
axes[2].set_title(f'Answer Length Distribution (answerable only)\n'
                  f'(mean={np.mean(a_words_all):.1f}, median={np.median(a_words_all):.0f})')
axes[2].axvline(np.mean(a_words_all), color='red', linestyle='--', linewidth=1.5)

plt.tight_layout()
fig.savefig(str(FIG_DIR / 'fig3_sources_length.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {FIG_DIR}/fig3_sources_length.png')
print('\nEvidence source counts:', dict(src_counts.most_common()))
print(f'Questions with no evidence source annotation: '
      f"{sum(len(q['_evidence_sources'])==0 for q in all_questions)}")

In [ ]:
# ─── Figure 4: Unanswerable rate by doc_type (top categories) ────────────
top_doc_types = [t for t, _ in all_type_counts.most_common(8)]
unans_by_type = {}
for dt in top_doc_types:
    qs = [q for q in all_questions if q['doc_type'] == dt]
    unans = sum(q['_unanswerable'] for q in qs)
    unans_by_type[shorten(dt, 26)] = (unans, len(qs) - unans, len(qs))

labels   = list(unans_by_type.keys())
unans_v  = [v[0] for v in unans_by_type.values()]
ans_v    = [v[1] for v in unans_by_type.values()]
totals   = [v[2] for v in unans_by_type.values()]

fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(labels))
width = 0.55
b1 = ax.bar(x, ans_v,   width, label='Answerable',   color='steelblue')
b2 = ax.bar(x, unans_v, width, bottom=ans_v,
            label='Unanswerable', color='salmon')

ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=28, ha='right', fontsize=9)
ax.set_ylabel('Number of questions')
ax.set_title('Answerable vs Unanswerable by Document Type (top 8 categories)')
ax.legend()

for i, (a, u, tot) in enumerate(zip(ans_v, unans_v, totals)):
    if u > 0:
        ax.text(i, tot + 1, f'{100*u/tot:.0f}%\nunanswer', ha='center', fontsize=7.5)

plt.tight_layout()
fig.savefig(str(FIG_DIR / 'fig4_unanswerable_by_type.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {FIG_DIR}/fig4_unanswerable_by_type.png')

In [ ]:
# ─── Figure 5: Cross-tab heatmap — doc_type × answer_format ─────────────
top8_types  = [t for t, _ in all_type_counts.most_common(8)]
all_formats = sorted({q.get('answer_format','Unknown') for q in all_questions})

xtab = pd.crosstab(
    pd.Series([shorten(q['doc_type'],26) for q in all_questions], name='doc_type'),
    pd.Series([q.get('answer_format','Unknown') for q in all_questions], name='answer_format')
)
# Keep only top 8 doc types
top8_short = [shorten(t,26) for t in top8_types]
xtab = xtab.loc[[t for t in top8_short if t in xtab.index]]

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(xtab, annot=True, fmt='d', cmap='YlOrRd',
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Count'})
ax.set_title('Cross-tabulation: Document Type × Answer Format (top 8 doc types)')
ax.set_xlabel('Answer Format')
ax.set_ylabel('Document Type')
plt.xticks(rotation=30, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
fig.savefig(str(FIG_DIR / 'fig5_crosstab_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {FIG_DIR}/fig5_crosstab_heatmap.png')
print('\n=== Cross-tabulation: doc_type × answer_format ===')
print(xtab.to_string())

In [ ]:
# ─── Figure 6: Evidence page number distribution (which pages cited most) ─
all_ev_page_nums = [p for q in all_questions for p in q['_evidence_pages']]
page_counter = Counter(all_ev_page_nums)
top_pages = page_counter.most_common(30)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
fig.suptitle('MMLongBench-Doc — Evidence Page Analysis', fontsize=13)

# Top 30 most cited page numbers
pg_nums = [p for p, _ in top_pages]
pg_vals = [v for _, v in top_pages]
axes[0].bar(range(len(pg_nums)), pg_vals, color='teal', edgecolor='white')
axes[0].set_xticks(range(len(pg_nums)))
axes[0].set_xticklabels([str(p) for p in pg_nums], rotation=60, fontsize=8)
axes[0].set_xlabel('Page number')
axes[0].set_ylabel('Citation count')
axes[0].set_title('Top 30 Most Cited Evidence Page Numbers')

# Cumulative distribution: how many questions need at most k pages
ev_counts_sorted = sorted(ev_page_counts)
cumulative = np.cumsum([1 for _ in ev_counts_sorted]) / N_ALL * 100
axes[1].plot(ev_counts_sorted, cumulative, color='darkblue', linewidth=2)
axes[1].fill_between(ev_counts_sorted, cumulative, alpha=0.15, color='blue')
axes[1].set_xlabel('Number of evidence pages')
axes[1].set_ylabel('Cumulative % of questions')
axes[1].set_title('Cumulative Distribution of Evidence Page Count')
axes[1].set_ylim(0, 102)
# Mark key thresholds
for threshold in [1, 2, 3]:
    pct = sum(c <= threshold for c in ev_page_counts) / N_ALL * 100
    axes[1].axvline(threshold, color='red', linestyle='--', alpha=0.5, linewidth=1)
    axes[1].text(threshold + 0.05, pct - 5, f'≤{threshold}pg:\n{pct:.1f}%',
                 fontsize=8, color='darkred')

plt.tight_layout()
fig.savefig(str(FIG_DIR / 'fig6_evidence_page_dist.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {FIG_DIR}/fig6_evidence_page_dist.png')
print(f'\nTop 10 most cited page numbers: {top_pages[:10]}')

In [ ]:
# ─── Figure 7: Pilot vs full corpus comparison ────────────────────────────
metrics_compare = {
    'Free-form fmt (%)': [
        100*N_FREEFORM/N_ALL,
        100*p_freeform/len(pilot_questions)
    ],
    'Cross-page (%)': [
        100*N_CROSS/N_ALL,
        100*n_crosspage/len(pilot_questions)
    ],
    'Visual evidence (%)': [
        100*N_VIS/N_ALL,
        100*n_visual/len(pilot_questions)
    ],
    'Mean evidence pages': [
        MEAN_EV,
        np.mean([q['_n_ev_pages'] for q in pilot_questions])
    ],
    'Mean Q length (words)': [
        MEAN_QW,
        np.mean([q['_q_words'] for q in pilot_questions])
    ],
}

fig, ax = plt.subplots(figsize=(11, 4.5))
met_names = list(metrics_compare.keys())
full_vals  = [metrics_compare[m][0] for m in met_names]
pilot_vals = [metrics_compare[m][1] for m in met_names]

x = np.arange(len(met_names))
w = 0.35
ax.bar(x - w/2, full_vals,  w, label='Full corpus (1,091)', color='steelblue')
ax.bar(x + w/2, pilot_vals, w, label=f'Pilot (N={len(pilot_questions)})', color='coral')

ax.set_xticks(x)
ax.set_xticklabels(met_names, rotation=20, ha='right')
ax.set_ylabel('Value')
ax.set_title('Pilot vs Full Corpus: Key Characteristics Comparison')
ax.legend()

for xi, (fv, pv) in enumerate(zip(full_vals, pilot_vals)):
    ax.text(xi - w/2, fv + 0.3, f'{fv:.1f}', ha='center', fontsize=8)
    ax.text(xi + w/2, pv + 0.3, f'{pv:.1f}', ha='center', fontsize=8)

plt.tight_layout()
fig.savefig(str(FIG_DIR / 'fig7_pilot_vs_full.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {FIG_DIR}/fig7_pilot_vs_full.png')

In [ ]:
# ─── Table: Sample QA pairs (5 per answer format) ────────────────────────
print('=== Sample QA Pairs by Answer Format ===')
for fmt in sorted({q.get('answer_format','Unknown') for q in all_questions}):
    sample_qs = [q for q in all_questions
                 if q.get('answer_format') == fmt and not q['_unanswerable']]
    if not sample_qs:
        continue
    random.seed(SEED)
    sample = random.sample(sample_qs, min(3, len(sample_qs)))
    print(f'\n--- Format: {fmt} ({len(sample_qs)} questions) ---')
    rows = []
    for q in sample:
        rows.append({
            'Question': q['question'][:80] + ('…' if len(q['question'])>80 else ''),
            'Answer': str(q['answer'])[:50] + ('…' if len(str(q['answer']))>50 else ''),
            'Ev. pages': str(q['_evidence_pages']),
            'Sources': str(q['_evidence_sources']),
        })
    print(pd.DataFrame(rows).to_string(index=False))

print('\n=== EDA complete. Figures saved to results/figures/ ===')

## Phase 2 — Verify PDF Availability

Reads PDFs from `data/mmlongbench/pdfs/` (flat) and the nested `hf_raw/documents/` subfolder.  
Expected: 134–135 PDFs total (one file has a very long filename and may be absent — safe to continue).  
If running inside Docker, PDFs are pre-mounted at the path above.

In [ ]:
import shutil

hf_raw_dir    = PDF_DIR / 'hf_raw'
existing_pdfs = list(PDF_DIR.glob('*.pdf'))
hf_pdfs       = list(hf_raw_dir.rglob('*.pdf')) if hf_raw_dir.exists() else []

print(f'PDFs in flat dir ({PDF_DIR.name}/):  {len(existing_pdfs)}')
print(f'PDFs in hf_raw/**:                  {len(hf_pdfs)}')
print(f'Total sources:                      {len(existing_pdfs) + len(hf_pdfs)}')

if not existing_pdfs and not hf_pdfs:
    print()
    print('WARNING: No PDFs found in either location.')
    print(f'  Expected flat dir: {PDF_DIR}')
    print(f'  Expected nested:   {hf_raw_dir}')
    print('Phases 3-9 will produce no output without PDFs.')
else:
    print('PDF sources ready.')

In [ ]:
doc_id_to_pdf = {}

for p in PDF_DIR.glob('*.pdf'):
    doc_id_to_pdf[p.name] = p

if hf_raw_dir.exists():
    for p in hf_raw_dir.rglob('*.pdf'):
        fname = p.name
        if fname not in doc_id_to_pdf:
            dest = PDF_DIR / fname
            if not dest.exists():
                try:
                    shutil.copy2(str(p), str(dest))
                except Exception as _cp_err:
                    logger.warning(f'copy2 failed for {p.name}: {_cp_err}')
                    continue
            doc_id_to_pdf[fname] = dest

all_doc_ids = {q['doc_id'] for q in all_questions}
found_all   = {d for d in all_doc_ids if d in doc_id_to_pdf}
missing     = all_doc_ids - found_all

print(f'PDFs found (unique): {len(doc_id_to_pdf)}')
print(f'Docs with PDFs     : {len(found_all)}/{len(all_doc_ids)}')
if missing:
    print(f'Missing ({len(missing)}): {sorted(missing)[:5]}{" ..." if len(missing)>5 else ""}')

parseable_doc_ids = [d for d in pilot_doc_ids if d in doc_id_to_pdf]
print(f'Parseable docs: {len(parseable_doc_ids)}')

## Phase 3 — Parse PDFs with MinerU + Temporal Annotation

MinerU (`magic-pdf`) converts each PDF into a structured JSON with:
bounding boxes, element types (text/table/figure/equation/section), reading order.

Install if needed: `pip install magic-pdf[full]`  
**Runtime:** ~30 s/doc × 12–20 pilot docs ≈ 10 min.

In [ ]:
from src.parsing.mineru_wrapper import parse_pdf

def parse_pilot_docs(parseable_doc_ids, doc_id_to_pdf, PARSED_DIR, PDF_DIR):
    parsed_count = skip_count = fail_count = 0
    for i, doc_id in enumerate(parseable_doc_ids, 1):
        out_path = PARSED_DIR / f'{doc_id}.json'
        if out_path.exists():
            skip_count += 1
            continue
        pdf_path = doc_id_to_pdf.get(doc_id)
        if not pdf_path or not Path(pdf_path).exists():
            logger.warning(f'[{i}] PDF not found: {doc_id}')
            fail_count += 1
            continue
        logger.info(f'[{i}/{len(parseable_doc_ids)}] Parsing: {doc_id}')
        try:
            art_dir = PARSED_DIR / 'artifacts' / Path(doc_id).stem
            result  = parse_pdf(str(pdf_path), str(art_dir))
            result.setdefault('doc_id', Path(doc_id).stem)
            out_path.write_text(json.dumps(result, indent=2), encoding='utf-8')
            parsed_count += 1
        except ImportError as e:
            print(f'⚠ MinerU not installed: {e}\n  pip install magic-pdf[full]')
            break
        except Exception as e:
            logger.error(f'Parse failed {doc_id}: {e}')
            fail_count += 1
    return parsed_count, skip_count, fail_count

parsed_count, skip_count, fail_count = parse_pilot_docs(
    parseable_doc_ids, doc_id_to_pdf, PARSED_DIR, PDF_DIR
)
print(f'Parsed: {parsed_count}, Skipped (cached): {skip_count}, Failed: {fail_count}')

parsed_ids = {p.stem for p in PARSED_DIR.glob('*.json') if p.stem != 'artifacts'}
print(f'Total parsed docs available: {len(parsed_ids)}')

In [ ]:
from src.parsing.temporal import annotate_parsed_elements

annotated = 0
for pf in sorted(PARSED_DIR.glob('*.json')):
    if pf.stem == 'artifacts':
        continue
    try:
        parsed = json.loads(pf.read_text(encoding='utf-8'))
        if any('temporal_markers' in e for e in parsed.get('elements', [])):
            continue
        parsed = annotate_parsed_elements(parsed, None, use_llm=False)
        pf.write_text(json.dumps(parsed, indent=2), encoding='utf-8')
        annotated += 1
    except Exception as e:
        logger.warning(f'Temporal annotation failed {pf.name}: {e}')

print(f'Temporal annotation: {annotated} docs annotated')

## Phase 4 — Node Embeddings + PyG Graph Construction

**Text nodes:** E5-Mistral-7B-Instruct (4-bit, ~4 GB VRAM) → 256-dim projection.  
**Image nodes:** SigLIP-SO400M (~1.5 GB VRAM) → 256-dim projection.  
Both run sequentially to stay within 8 GB VRAM.

**KG triplets:** GPT-4o-mini extraction (skipped if `OPENAI_API_KEY` not set).  
**Runtime:** ~2 min/doc × 12–20 docs.

In [ ]:
from src.graph.embeddings import compute_and_save_embeddings

embedded_count = skip_emb_count = 0

for pf in sorted(PARSED_DIR.glob('*.json')):
    if pf.stem == 'artifacts':
        continue
    doc_id = pf.stem
    text_pt = EMB_DIR / f'{doc_id}_text.pt'
    img_pt  = EMB_DIR / f'{doc_id}_img.pt'
    raw_pt  = EMB_DIR / f'{doc_id}_text_raw.pt'
    # Require raw E5 too: docs embedded before Option A lack _text_raw.pt
    # and must be recomputed for semantic vector retrieval.
    if text_pt.exists() and img_pt.exists() and raw_pt.exists():
        skip_emb_count += 1
        continue
    try:
        parsed = json.loads(pf.read_text(encoding='utf-8'))
        compute_and_save_embeddings(
            parsed,
            embeddings_dir=str(EMB_DIR),
            image_root=str(PDF_DIR / 'artifacts' / doc_id),
            device=DEVICE,
            text_batch_size=16,
            image_batch_size=8,
        )
        embedded_count += 1
    except Exception as e:
        logger.error(f'Embedding failed {doc_id}: {e}')

print(f'Embeddings computed: {embedded_count}, Skipped (cached): {skip_emb_count}')

In [ ]:
from src.graph.edges import extract_all_triplets
from src.graph.builder import build_graph, save_graph

openai_client_kg = None
if os.environ.get('OPENAI_API_KEY'):
    try:
        import openai
        openai_client_kg = openai.OpenAI(api_key=os.environ['OPENAI_API_KEY'])
        print('OpenAI client ready for KG extraction')
    except ImportError:
        print('openai package not found — KG triplets empty')
else:
    print('No OPENAI_API_KEY — KG triplets empty (graph still built)')

graph_count = skip_graph = 0

for pf in sorted(PARSED_DIR.glob('*.json')):
    if pf.stem == 'artifacts':
        continue
    doc_id = pf.stem
    if (GRAPH_DIR / f'{doc_id}.pt').exists():
        skip_graph += 1
        continue
    # Graph text features = RAW E5 (un-projected). The HGT's trainable
    # node_lin (Linear(-1, 256)) becomes the projection, learned in Phase 5
    # -- replacing the old random/discarded projection so GRAPH retrieval is
    # semantic, not noise.
    text_pt = EMB_DIR / f'{doc_id}_text_raw.pt'
    img_pt  = EMB_DIR / f'{doc_id}_img.pt'
    if not (text_pt.exists() and img_pt.exists()):
        logger.warning(f'Embeddings missing for {doc_id}')
        continue
    try:
        parsed = json.loads(pf.read_text(encoding='utf-8'))
        parsed.setdefault('doc_id', doc_id)
        embeddings = {
            'text': torch.load(text_pt, weights_only=True),
            'img':  torch.load(img_pt,  weights_only=True),
        }
        triplets = []
        if openai_client_kg:
            try:
                triplets = extract_all_triplets(parsed, openai_client_kg)
            except Exception as e:
                logger.warning(f'KG extraction failed {doc_id}: {e}')
        graph = build_graph(doc_id, parsed, embeddings, triplets)
        save_graph(graph, str(GRAPH_DIR), doc_id)
        graph_count += 1
    except Exception as e:
        logger.error(f'Graph build failed {doc_id}: {e}')

print(f'Graphs built: {graph_count}, Skipped (cached): {skip_graph}')
print(f'Total graphs available: {len(list(GRAPH_DIR.glob("*.pt")))}')

## Retrieval Helpers

`embed_query` embeds a question with the SAME E5 model/space the raw node
embeddings use. `node_pages` maps retrieved nodes to their page numbers for
APPA. `rerank_by_raw_e5` reorders by question↔node cosine. `page_context_nodes`
feeds the LLM full pages (reading order). `cross_rerank` (optional) jointly
scores (question, text) with a cross-encoder. Defined here so Phase 5 (HGT
training anchor), Phase 8 (vector) and Phase 9 (GraMM) all share one impl.

In [ ]:
from src.graph.embeddings import load_text_model
import torch.nn.functional as _Fnn
import numpy as _np

_QUERY_MODEL = None
def embed_query(question: str):
    global _QUERY_MODEL
    if _QUERY_MODEL is None:
        _QUERY_MODEL, _ = load_text_model(use_4bit=(DEVICE == 'cuda'), device=DEVICE)
    q = f'Represent this question for retrieving relevant document passages: {question}'
    # convert_to_tensor=False -> numpy -> fresh torch tensor (avoids
    # SentenceTransformer inference-mode tensors that break HGT backprop).
    emb = _QUERY_MODEL.encode([q], normalize_embeddings=True)
    return torch.from_numpy(_np.asarray(emb[0], dtype=_np.float32))

def node_pages(nodes, parsed):
    """Retrieved node dicts -> ordered list of page_no (int). Used for APPA."""
    if not parsed:
        return []
    type_to_elems = {}
    for e in parsed.get('elements', []):
        type_to_elems.setdefault(e['type'], []).append(e)
    pages = []
    for nd in nodes:
        nt, li = nd.get('node_type'), nd.get('local_idx')
        if nt is None or li is None:
            continue
        elems = type_to_elems.get(nt, [])
        if 0 <= li < len(elems):
            try:
                pages.append(int(elems[li].get('page_no', -1)))
            except (TypeError, ValueError):
                pass
    return pages

_RAWVEC_CACHE = {}
def _raw_node_vectors(doc_id, parsed):
    if doc_id in _RAWVEC_CACHE:
        return _RAWVEC_CACHE[doc_id]
    p = EMB_DIR / f'{doc_id}_text_raw.pt'
    out = {}
    if p.exists():
        te = torch.load(p, weights_only=True).float()
        from collections import Counter as _C
        tc = _C(e['type'] for e in parsed.get('elements', []))
        cur = 0
        for nt, cnt in [('text', tc.get('text', 0)),
                         ('section', tc.get('section', 0)),
                         ('equation', tc.get('equation', 0))]:
            if cnt > 0 and cur < te.shape[0]:
                end = min(cur + cnt, te.shape[0])
                for li in range(end - cur):
                    out[(nt, li)] = te[cur + li]
                cur = end
    _RAWVEC_CACHE[doc_id] = out
    return out

def rerank_by_raw_e5(nodes, q_vec, doc_id, parsed):
    if not nodes:
        return nodes
    vmap = _raw_node_vectors(doc_id, parsed or {'elements': []})
    qn = _Fnn.normalize(q_vec.float().unsqueeze(0), dim=-1)
    scored = []
    for nd in nodes:
        v = vmap.get((nd.get('node_type'), nd.get('local_idx')))
        if v is not None:
            s = float(_Fnn.cosine_similarity(qn, v.unsqueeze(0)).item())
        else:
            s = -1.0 + float(nd.get('score', 0.0)) * 1e-3
        nd['rerank_score'] = s
        scored.append((s, nd))
    scored.sort(key=lambda t: t[0], reverse=True)
    return [nd for _, nd in scored]

def page_context_nodes(ranked_nodes, parsed, n_pages):
    """Top-`n_pages` pages by PAGE_SCORE_MODE; emit synthetic node dicts for
    ALL elements on those pages in reading order (build_prompt re-resolves text)."""
    elements = (parsed or {}).get('elements', [])
    if not elements:
        return ranked_nodes
    type_to_elems = {}
    for e in elements:
        type_to_elems.setdefault(e['type'], []).append(e)
    page_scores, first_seen = {}, []
    for nd in ranked_nodes:
        nt, li = nd.get('node_type'), nd.get('local_idx')
        elems = type_to_elems.get(nt, [])
        if li is None or not (0 <= li < len(elems)):
            continue
        try:
            pg = int(elems[li].get('page_no', -1))
        except (TypeError, ValueError):
            continue
        sc = float(nd.get('rerank_score', nd.get('score', 0.0)))
        if pg not in page_scores:
            page_scores[pg] = []
            first_seen.append(pg)
        page_scores[pg].append(sc)
    if not page_scores:
        return ranked_nodes
    if PAGE_SCORE_MODE == 'first_seen':
        page_order = first_seen[:n_pages]
    else:
        def _agg(scs):
            if PAGE_SCORE_MODE == 'sum': return sum(scs)
            t = sorted(scs, reverse=True)[:max(1, PAGE_SCORE_TOPK)]
            return sum(t) / len(t)
        page_order = sorted(first_seen,
                            key=lambda p: (-_agg(page_scores[p]),
                                           first_seen.index(p)))[:n_pages]
    sel = set(page_order)
    out = []
    for nt, elems in type_to_elems.items():
        for li, e in enumerate(elems):
            try:
                pg = int(e.get('page_no', -1))
            except (TypeError, ValueError):
                continue
            if pg in sel:
                try:
                    ro = int(e.get('reading_order', 0))
                except (TypeError, ValueError):
                    ro = 0
                out.append((page_order.index(pg), ro,
                            {'node_type': nt, 'local_idx': li}))
    out.sort(key=lambda t: (t[0], t[1]))
    return [nd for _, _, nd in out]

_CE_MODEL = None
def cross_rerank(nodes, question, doc_id, parsed):
    if not nodes:
        return nodes
    global _CE_MODEL
    if _CE_MODEL is None:
        from sentence_transformers import CrossEncoder
        _CE_MODEL = CrossEncoder(CROSS_ENCODER_MODEL, device=DEVICE,
                                 max_length=512)
    elements = (parsed or {}).get('elements', [])
    type_to_elems = {}
    for e in elements:
        type_to_elems.setdefault(e['type'], []).append(e)
    pairs, idxs = [], []
    for j, nd in enumerate(nodes):
        nt, li = nd.get('node_type'), nd.get('local_idx')
        elems = type_to_elems.get(nt, [])
        txt = (elems[li].get('text', '') if li is not None
               and 0 <= li < len(elems) else nd.get('text', ''))
        if txt:
            pairs.append([question, txt[:2000]])
            idxs.append(j)
    if not pairs:
        return nodes
    scores = _CE_MODEL.predict(pairs, batch_size=32, show_progress_bar=False)
    for j, sc in zip(idxs, scores):
        nodes[j]['rerank_score'] = float(sc)
    for nd in nodes:
        nd.setdefault('rerank_score', -1e9)
    return sorted(nodes, key=lambda n: n['rerank_score'], reverse=True)

print('Retrieval helpers ready (embed_query, node_pages, '
      'rerank_by_raw_e5, page_context_nodes, cross_rerank).')

## Phase 5 — Train HGT with Evidence-Guided InfoNCE Loss

Uses MMLongBench `evidence_pages` annotations to create supervised training triples:
- **Anchor:** real question via `embed_query` -> `model.encode_query` (trained
  through the HGT's node_lin -- replaces the doc-mean proxy)
- **Positive:** nodes on annotated evidence pages
- **Negatives:** 15 random non-evidence nodes

Saved to `results/models/hgt_mmlb/best_model.pt`.

In [ ]:
from src.retrieval.hgt_model import DocumentHGT, info_nce_loss, METADATA
from src.graph.builder import load_graph

def get_evidence_node_indices(parsed, evidence_pages_set, node_type):
    typed = [e for e in parsed['elements'] if e['type'] == node_type]
    return [i for i, e in enumerate(typed)
            if e.get('page_no', -1) in evidence_pages_set]

def build_hgt_triple(item, model, GRAPH_DIR, PARSED_DIR, device):
    doc_id   = Path(item['doc_id']).stem
    ev_pages = set(item['_evidence_pages'])
    graph    = load_graph(str(GRAPH_DIR), doc_id)
    if graph is None:
        return None
    pf = PARSED_DIR / f'{doc_id}.json'
    if not pf.exists():
        return None
    parsed = json.loads(pf.read_text(encoding='utf-8'))

    x_dict = {nt: graph[nt].x.to(device)
              for nt in graph.node_types
              if hasattr(graph[nt], 'x') and graph[nt].x.shape[0] > 0}
    edge_index_dict = {et: graph[et].edge_index.to(device)
                       for et in graph.edge_types
                       if graph[et].edge_index.shape[1] > 0}
    if not x_dict:
        return None

    out_dict = model(x_dict, edge_index_dict)
    text_out = out_dict.get('text')
    if text_out is None or text_out.shape[0] == 0:
        return None
    # Anchor = REAL question via E5 + encode_query (trained through node_lin).
    # The model() call above initialised node_lin so encode_query can use it.
    # Trains the HGT to align question space with answer-evidence-page nodes,
    # replacing the doc-mean proxy that never saw the question.
    q_raw = embed_query(item['question']).to(device)
    q_emb = model.encode_query(q_raw)

    pos_embs = []
    for nt, embs in out_dict.items():
        for idx in get_evidence_node_indices(parsed, ev_pages, nt):
            if idx < embs.shape[0]:
                pos_embs.append(torch.nn.functional.normalize(embs[idx], dim=-1))

    all_embs = torch.cat([e for e in out_dict.values() if e.shape[0] > 0])
    if all_embs.shape[0] < 2:
        return None
    if not pos_embs:
        pos_emb = torch.nn.functional.normalize(
            all_embs[random.randint(0, all_embs.shape[0]-1)], dim=-1)
    else:
        pos_emb = random.choice(pos_embs)

    n_neg  = min(15, all_embs.shape[0] - 1)
    perm   = torch.randperm(all_embs.shape[0])[:n_neg]
    neg_embs = torch.nn.functional.normalize(all_embs[perm], dim=-1)
    return q_emb, pos_emb, neg_embs

print('HGT training helpers defined.')

In [ ]:
random.seed(SEED)
torch.manual_seed(SEED)

available_graph_ids = [
    p.stem for p in GRAPH_DIR.glob('*.pt')
    if p.stem in {Path(d).stem for d in pilot_doc_ids}
]

doc_to_questions = defaultdict(list)
for q in pilot_questions:
    stem = Path(q['doc_id']).stem
    if stem in set(available_graph_ids):
        doc_to_questions[stem].append(q)

print(f'HGT_EPOCHS = {HGT_EPOCHS}')
print(f'Training graphs available: {len(available_graph_ids)}')
print(f'Docs with evidence triples: {len(doc_to_questions)}')

if hgt_save_path.exists():
    print(f'HGT already trained: {hgt_save_path}')
elif not available_graph_ids:
    print('No graphs available. Run Phase 4 first to build graphs.')
else:
    hgt_model = DocumentHGT(metadata=METADATA).to(DEVICE)
    optimizer = torch.optim.Adam(hgt_model.parameters(), lr=1e-3, weight_decay=1e-4)
    best_loss = float('inf')

    for epoch in range(HGT_EPOCHS):
        hgt_model.train()
        total_loss = n_batches = 0
        all_qs = [q for qs in doc_to_questions.values() for q in qs]
        random.shuffle(all_qs)
        for item in all_qs:
            triple = build_hgt_triple(item, hgt_model, GRAPH_DIR, PARSED_DIR, DEVICE)
            if triple is None:
                continue
            q_emb, pos_emb, neg_embs = triple
            optimizer.zero_grad()
            loss = info_nce_loss(q_emb, pos_emb, neg_embs)
            if not (torch.isnan(loss) or torch.isinf(loss)):
                loss.backward()
                optimizer.step()
                total_loss += loss.item()
                n_batches  += 1
        avg_loss = total_loss / max(n_batches, 1)
        print(f'Epoch {epoch+1:3d}/{HGT_EPOCHS}  loss={avg_loss:.4f}  batches={n_batches}')
        if avg_loss < best_loss:
            best_loss = avg_loss
            torch.save(hgt_model.state_dict(), hgt_save_path)

    print(f'HGT training done. Best loss={best_loss:.4f}  Saved: {hgt_save_path}')

## Phase 6 — Train Query Router (DeBERTa-v3-base)

Auto-annotate routing labels from MMLongBench metadata:
- **GRAPH** — `evidence_pages` spans > 1 page (cross-page reasoning)
- **HYBRID** — single page with visual evidence (Chart / Figure / Table)
- **VECTOR** — single page, pure-text factoid

Fine-tunes `microsoft/deberta-v3-base`.  
Saved to `results/models/router_mmlb/best_model/`.

In [ ]:
from src.retrieval.router import build_router_training_data, LABEL2ID

all_for_router = []
for q in all_questions:
    q_copy = dict(q)
    q_copy['evidence_pages'] = q['_evidence_pages']  # list, not string
    all_for_router.append(q_copy)

samples = build_router_training_data(all_for_router, max_samples=len(all_for_router))

label_counts = Counter(s['label'] for s in samples)
print(f'Router training samples: {len(samples)}')
print(f'Label distribution: {dict(label_counts)}')

if len(samples) < 30:
    print('Augmenting with synthetic samples...')
    samples += [
        {'query': 'Compare trends across all sections',         'label': 'GRAPH'},
        {'query': 'What is the total revenue?',                 'label': 'VECTOR'},
        {'query': 'What does the chart on page 5 show?',        'label': 'HYBRID'},
        {'query': 'How did costs change over the years?',       'label': 'GRAPH'},
        {'query': 'What was net income in 2022?',               'label': 'VECTOR'},
        {'query': 'Summarize findings across all figures',      'label': 'GRAPH'},
        {'query': 'Which figure shows market share breakdown?', 'label': 'HYBRID'},
    ] * 15
    print(f'After augmentation: {len(samples)} samples')

In [ ]:
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer
)
import numpy as np

if router_save_dir.exists() and any(router_save_dir.iterdir()):
    print(f'Router already trained: {router_save_dir}')
else:
    print(f'ROUTER_EPOCHS = {ROUTER_EPOCHS}')
    ds = Dataset.from_dict({
        'text':  [s['query'] for s in samples],
        'label': [LABEL2ID[s['label']] for s in samples],
    })
    split    = ds.train_test_split(test_size=0.15, seed=SEED)
    train_ds = split['train']
    val_ds   = split['test']

    model_name = 'microsoft/deberta-v3-base'
    tokenizer  = AutoTokenizer.from_pretrained(model_name)

    def tokenize(batch):
        return tokenizer(batch['text'], truncation=True,
                         max_length=128, padding='max_length')

    train_enc = train_ds.map(tokenize, batched=True)
    val_enc   = val_ds.map(tokenize,   batched=True)

    router_model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=3,
        id2label={0:'GRAPH',1:'VECTOR',2:'HYBRID'},
        label2id={'GRAPH':0,'VECTOR':1,'HYBRID':2},
    )

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)
        return {'accuracy': float((preds == labels).mean())}

    training_args = TrainingArguments(
        output_dir=str(MODEL_DIR / 'router_mmlb'),
        num_train_epochs=ROUTER_EPOCHS,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        learning_rate=2e-5,
        weight_decay=0.01,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='accuracy',
        logging_steps=20,
        seed=SEED,
        report_to='none',
    )
    trainer = Trainer(
        model=router_model, args=training_args,
        train_dataset=train_enc, eval_dataset=val_enc,
        compute_metrics=compute_metrics,
    )
    trainer.train()
    trainer.save_model(str(router_save_dir))
    tokenizer.save_pretrained(str(router_save_dir))
    res = trainer.evaluate()
    print(f'Router val accuracy: {res.get("eval_accuracy",0):.4f}')
    print(f'Saved: {router_save_dir}')

## Phase 7 — Tune Reward Function (α, β, λ, τ Grid Search)

`R(q,K) = α·C + β·S + λ·T`  — C=coverage, S=similarity, T=temporal coherence

Grid search on a 20% holdout of pilot questions.  
Tuning signal: Spearman correlation between reward score and evidence-page hit.

In [ ]:
from src.retrieval.hgt_model import retrieve_top_k
from src.retrieval.self_healing import RewardFunction
import itertools

hgt_model_tune = DocumentHGT(metadata=METADATA).to(DEVICE)
if hgt_save_path.exists():
    hgt_model_tune.load_state_dict(
        torch.load(hgt_save_path, map_location=DEVICE, weights_only=False))
    print(f'HGT loaded: {hgt_save_path}')
else:
    print('HGT not trained yet — using random-init weights for reward tuning')
hgt_model_tune.eval()

random.seed(SEED + 1)
tuning_qs = random.sample(pilot_questions, max(4, int(0.2 * len(pilot_questions))))
print(f'Reward tuning on {len(tuning_qs)} held-out questions')

grid = [(a, b, l, t)
        for a, b, l, t in itertools.product(
            [0.30, 0.40, 0.50], [0.30, 0.35, 0.40],
            [0.20, 0.25, 0.30], [0.40, 0.50, 0.60, 0.70])
        if abs(a + b + l - 1.0) <= 0.05]
print(f'Grid size (weight-valid): {len(grid)} configurations')

if reward_save.exists():
    print(f'Reward params already tuned: {reward_save}')
else:
    best_score, best_params = -1.0, None
    for alpha, beta, lambda_, tau in grid:
        rf = RewardFunction(alpha=alpha, beta=beta, lambda_=lambda_, tau=tau)
        scores, hits = [], []
        for item in tuning_qs:
            doc_id = Path(item['doc_id']).stem
            graph  = load_graph(str(GRAPH_DIR), doc_id)
            pf     = PARSED_DIR / f'{doc_id}.json'
            if graph is None or not pf.exists():
                continue
            parsed = json.loads(pf.read_text(encoding='utf-8'))
            with torch.no_grad():
                node_emb_dict = hgt_model_tune.encode_nodes(
                    {nt: graph[nt].x.to(DEVICE) for nt in graph.node_types
                     if hasattr(graph[nt],'x') and graph[nt].x.shape[0]>0},
                    {et: graph[et].edge_index.to(DEVICE) for et in graph.edge_types
                     if graph[et].edge_index.shape[1]>0},
                )
            text_pt = EMB_DIR / f'{doc_id}_text.pt'
            q_emb = (torch.load(text_pt, weights_only=True).mean(0)
                     if text_pt.exists() else torch.zeros(256))
            nodes = retrieve_top_k(q_emb, node_emb_dict, top_k=10)
            rd = rf.compute(q_emb, nodes, node_emb_dict, parsed, [])
            scores.append(rd['reward'])
            ev_pages = set(item['_evidence_pages'])
            ret_pages = set()
            for n in nodes:
                typed = [e for e in parsed['elements'] if e['type']==n['node_type']]
                if n['local_idx'] < len(typed):
                    ret_pages.add(typed[n['local_idx']].get('page_no',-1))
            hits.append(1.0 if ev_pages and ev_pages & ret_pages else 0.0)
        if len(scores) > 2 and sum(hits) > 0:
            from scipy.stats import spearmanr
            corr, _ = spearmanr(scores, hits)
            corr = 0.0 if corr != corr else float(corr)
        else:
            corr = 0.0
        if corr > best_score:
            best_score = corr
            best_params = {'alpha':alpha,'beta':beta,'lambda_':lambda_,'tau':tau}

    if best_params is None:
        best_params = {'alpha':0.40,'beta':0.35,'lambda_':0.25,'tau':0.60}
        print('Grid search inconclusive. Using defaults.')
    reward_save.write_text(json.dumps(best_params, indent=2))
    print(f'Best reward params (corr={best_score:.3f}): {best_params}')

print('Reward tuning complete.')

## Phase 8 — Flat-Vector RAG Baseline

Baseline: FAISS vector search (no graph, no self-healing) + Qwen2.5-VL-72B.  
Uses the same VLM backend as GraMM-RAG to isolate retrieval contribution.

Results saved to `results/baseline_vector_mmlb_pilot.json`.

In [ ]:
from src.retrieval.vector_retrieval import VectorIndex
from src.generation.prompt_builder import build_prompt
from src.generation.vlm_client import VLMClient
from src.evaluation.metrics import compute_all_metrics
# embed_query / node_pages / rerank_by_raw_e5 / page_context_nodes /
# cross_rerank defined earlier (Retrieval Helpers cell).

faiss_indices = {}

# Build the FAISS index IN MEMORY from cached raw-E5 embeddings every run.
# We deliberately do NOT persist .index files -- the on-disk cache is the
# classic stale-cache trap (a file from an earlier broken run gets reloaded
# and returns garbage). _text_raw.pt is the real (expensive) cache.
_pilot_set = set(pilot_doc_ids) if N_PILOT else None
for pf in sorted(PARSED_DIR.glob('*.json')):
    if pf.stem == 'artifacts':
        continue
    doc_id = pf.stem
    if _pilot_set is not None and doc_id not in _pilot_set:
        continue
    text_pt = EMB_DIR / f'{doc_id}_text_raw.pt'
    img_pt  = EMB_DIR / f'{doc_id}_img.pt'
    if not text_pt.exists():
        continue
    try:
        parsed   = json.loads(pf.read_text(encoding='utf-8'))
        text_emb = torch.load(text_pt, weights_only=True).float()
        img_emb  = (torch.load(img_pt, weights_only=True).float()
                    if img_pt.exists() else torch.zeros(0, text_emb.shape[1]))
        elements = parsed.get('elements', [])
        tc = Counter(e['type'] for e in elements)
        node_emb_dict = {}
        cur = 0
        for nt, cnt in [('text',tc.get('text',0)),('section',tc.get('section',0)),
                         ('equation',tc.get('equation',0))]:
            if cnt > 0 and cur < text_emb.shape[0]:
                end = min(cur + cnt, text_emb.shape[0])
                node_emb_dict[nt] = text_emb[cur:end]
                cur = end
        cur = 0
        for nt, cnt in [('figure',tc.get('figure',0)),('table',tc.get('table',0))]:
            if cnt > 0 and img_emb.shape[0] > 0 and cur < img_emb.shape[0]:
                end = min(cur + cnt, img_emb.shape[0])
                node_emb_dict[nt] = img_emb[cur:end]
                cur = end
        if node_emb_dict:
            vi = VectorIndex()
            vi.build(node_emb_dict, parsed, doc_id)
            if vi.index is not None:
                faiss_indices[doc_id] = vi
    except Exception as e:
        logger.warning(f'FAISS build failed {doc_id}: {e}')

print(f'FAISS indices built (in-memory, from _text_raw.pt): {len(faiss_indices)}')

In [ ]:
if vector_out_path.exists():
    print(f'Flat-vector RAG results exist: {vector_out_path}')
    vector_results = json.loads(vector_out_path.read_text())
else:
    vlm_vec = VLMClient(provider='together', together_model='Qwen/Qwen2.5-VL-72B-Instruct')
    v_preds, v_golds, v_refused, v_answerable = [], [], [], []
    v_ret_pages, v_gold_pages = [], []

    def _ev_pages(it):
        ep = it.get('_evidence_pages') or it.get('evidence_pages') or []
        if isinstance(ep, str):
            try:    ep = ast.literal_eval(ep)
            except: ep = []
        return [int(p) for p in ep] if ep else []

    for i, item in enumerate(pilot_questions):
        question = item['question']
        gold     = item.get('answer', '')
        doc_id   = Path(item['doc_id']).stem

        q_emb = embed_query(question)   # raw E5 (semantic)

        vi    = faiss_indices.get(doc_id)
        _pool = RERANK_POOL if USE_CROSS_ENCODER else 10
        nodes = vi.search(q_emb, top_k=_pool) if vi and vi.index else []

        pf     = PARSED_DIR / f'{doc_id}.json'
        parsed = json.loads(pf.read_text(encoding='utf-8')) if pf.exists() else {'elements':[]}

        if USE_CROSS_ENCODER:
            ranked = cross_rerank(list(nodes), question, doc_id, parsed)
        else:
            ranked = sorted(nodes, key=lambda n: n.get('score', 0.0),
                            reverse=True)
        if PAGE_CONTEXT:
            gen_nodes = page_context_nodes(ranked, parsed, PAGE_CONTEXT_N)
            mcc = PAGE_CONTEXT_CHARS
        else:
            gen_nodes = ranked[:GEN_TOP_N]
            mcc = 4000

        prompt = build_prompt(question, gen_nodes, parsed, {}, False,
                              benchmark='mmlongbench', max_context_chars=mcc)
        answer = vlm_vec.generate(prompt, temperature=0.0, max_tokens=256)

        v_preds.append(answer)
        v_golds.append(str(gold) if gold is not None else '')
        v_refused.append(False)
        # MMLongBench HAS unanswerable Qs: gold=None / 'Not answerable'.
        v_answerable.append(gold is not None and str(gold).strip().lower()
                            not in ('', 'not answerable', 'none'))
        v_ret_pages.append(node_pages(nodes, parsed))
        v_gold_pages.append(_ev_pages(item))

        if (i + 1) % 10 == 0:
            print(f'  Vector RAG [{i+1}/{len(pilot_questions)}]')

    v_metrics = compute_all_metrics(v_preds, v_golds, v_answerable, v_refused,
                                    retrieved_pages=v_ret_pages,
                                    gold_pages=v_gold_pages)
    vector_results = {
        'system': 'Flat-vector RAG', 'n_questions': len(pilot_questions),
        'metrics': v_metrics, 'predictions': v_preds, 'golds': v_golds,
        'retrieved_pages': v_ret_pages, 'gold_pages': v_gold_pages,
        'is_answerable': v_answerable,
    }
    vector_out_path.write_text(json.dumps(vector_results, indent=2))
    print(f'Flat-vector RAG: ANLS={v_metrics["anls"]:.3f} '
          f'F1={v_metrics["f1"]:.3f} Acc={v_metrics["accuracy"]:.3f} '
          f'APPA@1={v_metrics.get("appa@1", float("nan")):.3f} '
          f'APPA@10={v_metrics.get("appa@10", float("nan")):.3f}')
    print(f'Saved: {vector_out_path}')

## Phase 9 — GraMM-RAG Full Evaluation (Single Seed)

Full pipeline: **route → retrieve (HGT/FAISS/hybrid) → self-heal → generate → score**

Single seed matches the deterministic inference config (temperature=0, fixed
weights); statistical rigor comes from paired bootstrap (1,000 resamples) in
the analysis, not seed averaging.

True-hybrid: HGT/self-healing provides graph recall; FAISS candidates are
MERGED into the page-selection pool so the answer node is reachable even
when graph expansion floods tangential pages.

In [ ]:
from src.retrieval.router import QueryRouter
from src.retrieval.self_healing import SelfHealingRetriever

hgt_model_eval = DocumentHGT(metadata=METADATA).to(DEVICE)
if hgt_save_path.exists():
    hgt_model_eval.load_state_dict(
        torch.load(hgt_save_path, map_location=DEVICE, weights_only=False))
    print(f'HGT loaded: {hgt_save_path}')
else:
    print('⚠ HGT weights not found — using random init')
hgt_model_eval.eval()

router = QueryRouter(
    model_dir=str(router_save_dir) if router_save_dir.exists() else None,
    device=DEVICE,
)
print(f'Router: {"DeBERTa" if router.model else "heuristic fallback"}')

reward_fn = RewardFunction(
    params_path=str(reward_save) if reward_save.exists() else None)

retriever = SelfHealingRetriever(
    hgt_model=hgt_model_eval, reward_fn=reward_fn,
    top_k=10, expansion_k=5, max_rounds=2,
)
vlm_eval = VLMClient(provider='together', together_model='Qwen/Qwen2.5-VL-72B-Instruct')
print('All GraMM-RAG components loaded.')

In [ ]:
def _ev_pages(it):
    ep = it.get('_evidence_pages') or it.get('evidence_pages') or []
    if isinstance(ep, str):
        try:    ep = ast.literal_eval(ep)
        except: ep = []
    return [int(p) for p in ep] if ep else []

def run_gramm_rag_mmlb(questions, seed):
    random.seed(seed); torch.manual_seed(seed)
    predictions, golds, refused_flags, reward_scores = [], [], [], []
    ret_pages, gold_pages, routes, answerable = [], [], [], []

    for i, item in enumerate(questions):
        question = item['question']
        gold     = item.get('answer', '')
        if isinstance(gold, list):
            gold = gold[0] if gold else ''
        gold_str = str(gold) if gold is not None else ''
        doc_id   = Path(item['doc_id']).stem

        graph  = load_graph(str(GRAPH_DIR), doc_id)
        pf     = PARSED_DIR / f'{doc_id}.json'
        parsed = json.loads(pf.read_text(encoding='utf-8')) if pf.exists() else None
        route  = router.route(question)

        # Raw E5 query for BOTH paths (self-healing projects via encode_query).
        q_vec = embed_query(question)
        vi = faiss_indices.get(doc_id)

        if graph and parsed and route in ('GRAPH', 'HYBRID'):
            result = retriever.retrieve(q_vec, graph, parsed, [])
            # TRUE HYBRID: merge raw-E5 FAISS candidates into page-selection
            # pool. Self-healing has page RECALL (high APPA) but the answer
            # node is rarely rank-1; FAISS adds vector precision so page
            # selection lands on the answer page. Reward/refused untouched.
            pool = list(result['nodes'])
            seen = {(n.get('node_type'), n.get('local_idx')) for n in pool}
            if vi and vi.index:
                for n in vi.search(q_vec, top_k=RERANK_POOL):
                    if (n.get('node_type'), n.get('local_idx')) not in seen:
                        pool.append(n)
        else:
            nodes = vi.search(q_vec, top_k=10) if vi and vi.index else []
            result = {'nodes':nodes,'reward':0.5,'refused':False,
                      'rounds':0,'reward_detail':{}}
            pool = list(nodes)

        # Precision rerank, then page-aware context (or top-N).
        if USE_CROSS_ENCODER:
            ranked = cross_rerank(pool, question, doc_id, parsed)
        else:
            ranked = rerank_by_raw_e5(pool, q_vec, doc_id, parsed)
        if PAGE_CONTEXT:
            gen_nodes = page_context_nodes(ranked, parsed or {'elements':[]},
                                           PAGE_CONTEXT_N)
            mcc = PAGE_CONTEXT_CHARS
        else:
            gen_nodes = ranked[:GEN_TOP_N]
            mcc = 4000

        prompt = build_prompt(
            question, gen_nodes,
            parsed or {'elements':[]},
            result['reward_detail'], result['refused'],
            benchmark='mmlongbench', max_context_chars=mcc,
        )
        answer = vlm_eval.generate(prompt, temperature=0.0, max_tokens=256)

        predictions.append(answer); golds.append(gold_str)
        refused_flags.append(result['refused'])
        reward_scores.append(result['reward'])
        # APPA on graph-retrieved set (pure graph-retrieval metric).
        ret_pages.append(node_pages(result['nodes'], parsed))
        gold_pages.append(_ev_pages(item))
        routes.append(route)
        # MMLongBench HAS unanswerable Qs (gold None / 'Not answerable').
        answerable.append(gold is not None and str(gold).strip().lower()
                          not in ('', 'not answerable', 'none'))

        if (i + 1) % 10 == 0:
            print(f'  [{i+1}/{len(questions)}] route={route} '
                  f'R={result["reward"]:.2f} refused={result["refused"]}')

    metrics = compute_all_metrics(predictions, golds, answerable,
                                  refused_flags,
                                  retrieved_pages=ret_pages,
                                  gold_pages=gold_pages)
    return {
        'n_questions':len(questions), 'metrics':metrics,
        'predictions':predictions, 'golds':golds,
        'reward_scores':reward_scores, 'refused_flags':refused_flags,
        'retrieved_pages':ret_pages, 'gold_pages':gold_pages,
        'routes':routes, 'is_answerable':answerable,
    }

print('run_gramm_rag_mmlb() defined.')

In [ ]:
gramm_results = {}

for seed in [SEED]:
    out_path = RESULTS_DIR / f'gramm_mmlongbench_s{seed}.json'
    if out_path.exists():
        print(f'Seed {seed}: results exist at {out_path}')
        gramm_results[seed] = json.loads(out_path.read_text())
        continue
    print(f'\n=== GraMM-RAG / seed={seed} ===')
    res = run_gramm_rag_mmlb(pilot_questions, seed=seed)
    gramm_results[seed] = res
    out_path.write_text(json.dumps(res, indent=2))
    m = res['metrics']
    print(f'  ANLS={m["anls"]:.3f}  F1={m["f1"]:.3f}  Acc={m["accuracy"]:.3f}  '
          f'APPA@1={m.get("appa@1",float("nan")):.3f} '
          f'APPA@10={m.get("appa@10",float("nan")):.3f}  '
          f'AbsF1={m.get("abstention_f1",0):.3f}  '
          f'Refused={sum(res["refused_flags"])}/{res["n_questions"]}')

## Phase 10 — Results & Analysis

Compare GraMM-RAG against the flat-vector RAG baseline (same single-seed,
deterministic protocol). Statistical rigor: paired bootstrap (1,000 resamples)
on a fixed-seed result -- *not* multi-seed averaging.

In [ ]:
import pandas as pd

rows = []
if vector_out_path.exists():
    vr = json.loads(vector_out_path.read_text())
    m  = vr['metrics']
    rows.append({
        'System': 'Flat-vector RAG', 'N': vr['n_questions'],
        'ANLS': f"{m['anls']:.3f}", 'F1': f"{m['f1']:.3f}",
        'Accuracy': f"{m['accuracy']:.3f}",
        'APPA@1':  f"{m.get('appa@1', float('nan')):.3f}",
        'APPA@5':  f"{m.get('appa@5', float('nan')):.3f}",
        'APPA@10': f"{m.get('appa@10', float('nan')):.3f}",
        'Abst-F1': f"{m.get('abstention_f1',0):.3f}",
        'Refused': 0, 'Seed': '-',
    })
for seed, res in sorted(gramm_results.items()):
    m = res['metrics']
    rows.append({
        'System': 'GraMM-RAG', 'N': res['n_questions'],
        'ANLS': f"{m['anls']:.3f}", 'F1': f"{m['f1']:.3f}",
        'Accuracy': f"{m['accuracy']:.3f}",
        'APPA@1':  f"{m.get('appa@1', float('nan')):.3f}",
        'APPA@5':  f"{m.get('appa@5', float('nan')):.3f}",
        'APPA@10': f"{m.get('appa@10', float('nan')):.3f}",
        'Abst-F1': f"{m.get('abstention_f1',0):.3f}",
        'Refused': sum(res['refused_flags']), 'Seed': seed,
    })
if rows:
    df = pd.DataFrame(rows)
    print('=== GraMM-RAG vs Flat-Vector RAG on MMLongBench-Doc ===')
    print(f'N={len(pilot_questions)} questions, Seed={SEED}, '
          'Model=Qwen2.5-VL-72B-Instruct')
    print(); print(df.to_string(index=False))
else:
    print('No results yet. Run Phases 8-9 first.')

### Table A — Per-Route Metric Matrix (GraMM vs Flat-Vector)

Route is a question-level property (same router decision for both systems),
so VECTOR/HYBRID/GRAPH/ALL splits are comparable. Deterministic columns
(APPA) are the robust GraMM signal; ANLS/F1/Acc are LLM-stochastic and at
pilot scale carry run-to-run noise -- read direction, confirm at full scale
with the bootstrap.

In [ ]:
from src.evaluation.metrics import (compute_anls, compute_f1,
    compute_accuracy, compute_answer_page_accuracy)

vr = json.loads(vector_out_path.read_text()) if vector_out_path.exists() else None
VP = vr['predictions'] if vr else None
VG = vr['golds'] if vr else None
V_RP = vr.get('retrieved_pages') if vr else None
V_GP = vr.get('gold_pages') if vr else None
V_AN = vr.get('is_answerable') if vr else None

def _metrics(P, G, RP, GP):
    m = {'ANLS': compute_anls(P, G), 'F1': compute_f1(P, G),
         'Acc': compute_accuracy(P, G)}
    if RP and GP:
        ap = compute_answer_page_accuracy(RP, GP)
        m['APPA@1'] = ap['appa@1']; m['APPA@10'] = ap['appa@10']
    else:
        m['APPA@1'] = m['APPA@10'] = float('nan')
    return m

for seed, res in sorted(gramm_results.items()):
    rts = res.get('routes')
    if not rts:
        print(f'seed {seed}: no route log'); continue
    P, G = res['predictions'], res['golds']
    RP, GP = res.get('retrieved_pages', []), res.get('gold_pages', [])
    rows = []
    for rt in ['VECTOR', 'HYBRID', 'GRAPH', 'ALL']:
        idx = (list(range(len(rts))) if rt == 'ALL'
               else [i for i, r in enumerate(rts) if r == rt])
        if not idx:
            continue
        gm = _metrics([P[i] for i in idx], [G[i] for i in idx],
                      [RP[i] for i in idx] if RP else None,
                      [GP[i] for i in idx] if GP else None)
        vm = (_metrics([VP[i] for i in idx], [VG[i] for i in idx],
                       [V_RP[i] for i in idx] if V_RP else None,
                       [V_GP[i] for i in idx] if V_GP else None)
              if VP else {k: float('nan') for k in gm})
        rows.append({
            'Route': rt, 'N': len(idx),
            'GraMM ANLS': f"{gm['ANLS']:.3f}", 'Vec ANLS': f"{vm['ANLS']:.3f}",
            'dANLS': f"{gm['ANLS']-vm['ANLS']:+.3f}",
            'GraMM F1': f"{gm['F1']:.3f}", 'GraMM Acc': f"{gm['Acc']:.3f}",
            'GraMM APPA@1':  f"{gm['APPA@1']:.3f}",
            'Vec APPA@1':    f"{vm['APPA@1']:.3f}",
            'GraMM APPA@10': f"{gm['APPA@10']:.3f}",
            'Vec APPA@10':   f"{vm['APPA@10']:.3f}",
        })
    print(f'=== Table A -- Per-Route (seed {seed}, N_PILOT={N_PILOT}) ===')
    print(pd.DataFrame(rows).to_string(index=False))

### Table B — Head-to-Head Exact-Match Contingency (McNemar)

Paired per-question exact-match outcomes; McNemar's chi-square tests whether
the discordant pairs are imbalanced. At pilot N this is usually n.s.;
informative at full scale (N=1,091).

In [ ]:
from src.evaluation.metrics import normalise_answer
vr = json.loads(vector_out_path.read_text()) if vector_out_path.exists() else None
def _ok(p, gold):
    pn = normalise_answer(p)
    gs = gold if isinstance(gold, list) else [gold]
    return any(pn == normalise_answer(g) for g in gs if g is not None)
for seed, res in sorted(gramm_results.items()):
    P, G = res['predictions'], res['golds']
    if not vr:
        print('No vector results to compare.'); break
    VP = vr['predictions']
    gok = [_ok(P[i], G[i]) for i in range(len(P))]
    vok = [_ok(VP[i], G[i]) for i in range(len(P))]
    both    = sum(1 for a, b in zip(gok, vok) if a and b)
    gonly   = sum(1 for a, b in zip(gok, vok) if a and not b)
    vonly   = sum(1 for a, b in zip(gok, vok) if b and not a)
    neither = sum(1 for a, b in zip(gok, vok) if not a and not b)
    n = len(P)
    mc = ((abs(gonly - vonly) - 1) ** 2) / (gonly + vonly) if (gonly + vonly) else 0.0
    try:
        from scipy.stats import chi2
        pval = float(chi2.sf(mc, 1))
    except Exception:
        pval = float('nan')
    print(f'=== Table B -- Exact-Match Contingency (seed {seed}, N={n}) ===')
    print(f'  both correct   : {both:>4}')
    print(f'  GraMM-only     : {gonly:>4}')
    print(f'  Vec-only       : {vonly:>4}')
    print(f'  neither        : {neither:>4}')
    print(f'  verdict rests on {gonly + vonly} discordant / {n} questions')
    print(f'  McNemar chi2={mc:.3f}  p={pval:.4f}  '
          f'({"significant" if pval < 0.05 else "n.s."} at 0.05)')

### Table C — Answer-Quality Buckets

`exact` (ANLS=1), `partial` (0<ANLS<1), `zero` (ANLS=0 non-refusal),
`refusal`. MMLongBench has unanswerable Qs, so refusal is sometimes correct
behaviour -- read in conjunction with Abstention-F1.

In [ ]:
from src.evaluation.metrics import anls_score, normalise_answer
_REF = ('no information', 'no relevant', 'does not contain', 'cannot',
        'not provide', 'unable', 'not answerable', 'insufficient',
        'no document')
def _bucket(pred, gold):
    gs = gold if isinstance(gold, list) else [gold]
    a = max(anls_score(pred, g) for g in gs if g is not None) if any(
        g is not None for g in gs) else 0.0
    if a >= 0.999: return 'exact'
    if a > 0.0:    return 'partial'
    if isinstance(pred, str) and any(t in pred.lower() for t in _REF):
        return 'refusal'
    return 'zero'
from collections import Counter as _C
rows = []
vr = json.loads(vector_out_path.read_text()) if vector_out_path.exists() else None
if vr:
    b = _C(_bucket(p, g) for p, g in zip(vr['predictions'], vr['golds']))
    rows.append({'System': 'Flat-vector', **{k: b.get(k, 0) for k in
                 ['exact','partial','zero','refusal']}})
for seed, res in sorted(gramm_results.items()):
    b = _C(_bucket(p, g) for p, g in zip(res['predictions'], res['golds']))
    rows.append({'System': f'GraMM s{seed}', **{k: b.get(k, 0) for k in
                 ['exact','partial','zero','refusal']}})
print('=== Table C -- Answer-Quality Buckets ===')
print(pd.DataFrame(rows).to_string(index=False))

### APPA Analysis — Answer-Page Prediction Accuracy

Left: APPA@k for GraMM vs Flat-Vector. Right: rank at which the first gold
answer page appears in the retrieved list (lower = better localisation;
'miss' = no gold page retrieved). MMLongBench's `evidence_pages` is a LIST
of pages -- APPA counts a hit if ANY gold page is in top-k.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
FIG_DIR = RESULTS_DIR / 'figures'; FIG_DIR.mkdir(parents=True, exist_ok=True)

def first_hit_rank(ret_pages, gold_pages):
    ranks = []
    for pages, gold in zip(ret_pages, gold_pages):
        gs = set(gold) if isinstance(gold, (list,set,tuple)) else {gold}
        r = next((j for j, p in enumerate(pages) if p in gs), None)
        ranks.append(r)
    return ranks

panels = []
if vector_out_path.exists():
    vr = json.loads(vector_out_path.read_text())
    panels.append(('Flat-vector RAG', vr['metrics'],
                   vr.get('retrieved_pages'), vr.get('gold_pages')))
for seed, res in sorted(gramm_results.items()):
    panels.append((f'GraMM-RAG (s{seed})', res['metrics'],
                   res.get('retrieved_pages'), res.get('gold_pages')))

ks = [1, 5, 10]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
width = 0.8 / max(len(panels), 1)
x = np.arange(len(ks))
for idx, (name, m, _rp, _gp) in enumerate(panels):
    vals = [m.get(f'appa@{k}', 0.0) for k in ks]
    ax1.bar(x + idx * width, vals, width, label=name)
ax1.set_xticks(x + width * (len(panels) - 1) / 2)
ax1.set_xticklabels([f'APPA@{k}' for k in ks])
ax1.set_ylabel('Accuracy'); ax1.set_ylim(0, 1)
ax1.set_title('Answer-Page Prediction Accuracy'); ax1.legend(fontsize=8)
ax1.grid(axis='y', alpha=0.3)
for name, _m, rp, gp in panels:
    if not rp or not gp: continue
    ranks = first_hit_rank(rp, gp)
    hit = [r for r in ranks if r is not None]
    miss = sum(1 for r in ranks if r is None)
    ax2.hist(hit, bins=range(0, 12), alpha=0.5,
             label=f'{name} (miss={miss}/{len(ranks)})')
ax2.set_xlabel('Rank of first gold answer page (0 = top)')
ax2.set_ylabel('# questions'); ax2.set_title('Gold-Page Retrieval Rank')
ax2.legend(fontsize=8); ax2.grid(axis='y', alpha=0.3)
fig.tight_layout()
fig.savefig(str(FIG_DIR / 'fig_appa_comparison.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {FIG_DIR}/fig_appa_comparison.png')

### Figure — Self-Healing Reward vs Answer Quality (GraMM)

Validates a core thesis contribution: does the self-healing reward track
answer quality? Per-question reward binned, ANLS distribution per bin. A
rising trend = reward is a meaningful retrieval-quality proxy.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from src.evaluation.metrics import anls_score
FIG_DIR = RESULTS_DIR / 'figures'; FIG_DIR.mkdir(parents=True, exist_ok=True)
for seed, res in sorted(gramm_results.items()):
    rw = res.get('reward_scores')
    if not rw:
        print(f'seed {seed}: no reward_scores'); continue
    P, G = res['predictions'], res['golds']
    per_anls = [max(anls_score(P[i], g) for g in
                    (G[i] if isinstance(G[i], list) else [G[i]])
                    if g is not None) if any(g is not None for g in
                    (G[i] if isinstance(G[i], list) else [G[i]])) else 0.0
                for i in range(len(P))]
    rw = np.array(rw[:len(per_anls)], dtype=float)
    pa = np.array(per_anls[:len(rw)], dtype=float)
    edges = np.unique(np.quantile(rw, [0, .2, .4, .6, .8, 1.0]))
    if len(edges) < 3:
        print(f'seed {seed}: reward has no spread '
              f'(min={rw.min():.3f} max={rw.max():.3f}); skipping plot')
        continue
    binned, labels = [], []
    for j in range(len(edges) - 1):
        lo, hi = edges[j], edges[j + 1]
        sel = (rw >= lo) & (rw <= hi if j == len(edges) - 2 else rw < hi)
        if sel.sum():
            binned.append(pa[sel]); labels.append(f'{lo:.2f}-{hi:.2f}')
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.boxplot(binned, labels=labels, showmeans=True)
    means = [b.mean() for b in binned]
    ax.plot(range(1, len(means) + 1), means, 'r--o', label='mean ANLS')
    ax.set_xlabel('Self-healing reward bin'); ax.set_ylabel('Per-question ANLS')
    ax.set_title(f'Reward vs Answer Quality (GraMM s{seed}, N={len(rw)})')
    ax.legend(); ax.grid(axis='y', alpha=0.3); fig.tight_layout()
    fig.savefig(str(FIG_DIR / f'fig_reward_anls_s{seed}.png'),
                dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {FIG_DIR}/fig_reward_anls_s{seed}.png  '
          f'(corr={np.corrcoef(rw, pa)[0,1]:+.3f})')

In [ ]:
summary = {
    'N_PILOT': N_PILOT, 'SEED': SEED,
    'model': 'Qwen/Qwen2.5-VL-72B-Instruct',
    'flat_vector_rag': json.loads(vector_out_path.read_text())['metrics']
        if vector_out_path.exists() else None,
    'gramm_rag': {str(s): r['metrics'] for s, r in gramm_results.items()},
}
(RESULTS_DIR / 'summary_mmlb.json').write_text(json.dumps(summary, indent=2))

print('All results saved to results/')
print()
print('Estimated API cost: ~$8 (Qwen2.5-VL-72B x single seed x 1,091 Qs)')